<center><h2><strong><font color="blue"> Advanced Programming for Data Science (APDS)</font></strong></h2></center>

<center><img alt="" src="images/covers/taudata-cover.jpg"/></center>

<center><h2><strong><font color="blue">APDS-13: GPU Programming in Python</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

## Outline

* Review What we have learnt and what we will learn
* Computer Architecture & GPU
* Latest Computational hardware Trends
* Setup GPU Environment (Google Colab & Local)
* GPU in Python: CuPy, Cuda, Numba, via other module (TensorFlow/PyTorch)
* Latency & Throughput
* Bottleneck: Data Transfer Overhead
* Case Study

<center><h2><strong><font color="blue"> Computer Architecture & Graphics Processing Unit </font></strong></h2></center>

### Conventional PC/Notebook

<center><img alt="" src="images/APDS/cpu-motherboard.jpg" style="height: 480px;"/></center>

* image source: AI Generated

<center><h2><strong><font color="blue"> PC, Workstation, & Server </font></strong></h2></center>

<center><img alt="" src="images/APDS/pc-workstation-server.jpg" style="height: 480px;"/></center>

* image source: Ai Generated

<center><h2><strong><font color="blue"> Workstation, & Server Motherboards</font></strong></h2></center>

<center><img alt="" src="images/APDS/workstation-server-motherboards.jpg" style="height: 480px;"/></center>

* image source: Various

<center><h2><strong><font color="blue"> Central Processing Unit (CPU)</font></strong></h2></center>

<center><img alt="" src="images/APDS/CPUs.jpg" style="height: 320px;"/></center>

* image source: Various

<center><h2><strong><font color="blue"> Graphics Processing Unit (GPU)</font></strong></h2></center>

> Originally designed for video games, the GPU's architecture—thousands of small, efficient cores—is mathematically perfect for the linear algebra operations that underpin modern Data Science.

<center><img alt="" src="images/APDS/GPUs.jpg" style="height: 320px;"/></center>

* image source: Various

<center><h2><strong><font color="blue"> Central Processing Unit VS Graphics Processing Unit </font></strong></h2></center>

<center><img alt="" src="images/APDS/cpu-vs-gpu.jpg" style="height: 480px;"/></center>

* image source: AI Generated

### Video Illustrations: https://www.youtube.com/watch?v=WmW6SD-EHVY

## 🧠 Theory: The CPU vs. GPU Analogy

### The CPU (Central Processing Unit)
* **Analogy:** A Ferrari.
* **Strength:** Extremely fast at doing *one thing* at a time (Low Latency).
* **Best for:** Sequential logic, branching (if/else), OS tasks.
* **Cores:** Few (4-64 powerful cores).

### The GPU (Graphics Processing Unit)
* **Analogy:** A Bus (or a fleet of buses).
* **Strength:** Slow at starting, but carries massive amounts of data at once (High Throughput).
* **Best for:** Parallel tasks (matrix math, image processing).
* **Cores:** Many (Thousands of smaller, weaker cores).

### Key Takeaway
**GPUs are not "faster" CPUs.** They are throughput engines. You should only use them when you can parallelize a task across thousands of data points.

<center><h2><strong><font color="blue"> Compute Nodes </font></strong></h2></center>

<center><img alt="" src="images/APDS/GPU-NPU-TPU-FPGA-DPU.jpg" style="height: 480px;"/></center>

* image source: AI Generated

<center><h2><strong><font color="blue"> CPU, GPU, and TPU in Google Collab </font></strong></h2></center>

**Enable the GPU**
1. Open a new notebook in Google Colab.
2. Go to Runtime > Change runtime type.
3. Under Hardware accelerator, select T4 GPU.
4. Click Save.

<center><img alt="" src="images/APDS/Google-Colab.jpg" style="height: 480px;"/></center>

### Also available for free in Kaggle, IBM Cloud, etc.

In [ ]:
# Check NVIDIA System Management Interface for GPU details
!nvidia-smi

## The Golden Rule of GPU Programming:

* The GPU has high throughput but high latency (start-up time).
* **Don't**: Move data back and forth between CPU and GPU for small calculations (e.g., adding two numbers). The transfer time will take longer than the math.
* **Don't**: Use GPU for small dataset.
* **Do**: Move a large chunk of data to the GPU once, perform millions of operations, and return only the result

<center><h2><strong><font color="blue"> GPU Programming Modules in Python </font></strong></h2></center>

<center><img alt="" src="images/APDS/GPU-Prog-modules-in-Python.jpg" style="height: 480px;"/></center>

* image source: AI Generated

# CuPy - NumPy on Steroids

### What is CuPy?
CuPy is a library that implements the NumPy array interface but runs on NVIDIA GPUs. 

<center><img alt="" src="images/APDS/cupy_logo_1000px.png" style="height: 250px;"/></center>

### https://github.com/cupy/cupy

# Numpy VS CuPy 

The computation below is executed on the **CPU** using NumPy.

For large arrays, this can be relatively slow.

In [1]:
import numpy as np

N = 10_000_000
a = np.random.rand(N)
b = np.random.rand(N)

c = a + b
c[:5]

array([0.52764653, 1.60270515, 1.25927427, 1.09045251, 0.62439713])

## 2. GPU Acceleration with CuPy

CuPy provides a NumPy-like API but executes operations on the GPU.

Here, the same operation runs on the **GPU**.

The syntax is nearly identical to NumPy.

In [ ]:
import cupy as cp

a_gpu = cp.random.rand(N)
b_gpu = cp.random.rand(N)

c_gpu = a_gpu + b_gpu
c_gpu[:5]

# Another example

In [ ]:
import numpy as np
import cupy as cp
import time

# 1. Array Creation
# Create a standard NumPy array (Exists in System RAM)
x_cpu = np.array([1, 2, 3])

# Create a CuPy array (Exists in GPU VRAM)
x_gpu = cp.array([1, 2, 3])

print("CPU Array Device:", x_cpu.dtype)
print("GPU Array Device:", x_gpu.device)

### ⚡ Live Benchmark: Matrix Multiplication
Let's multiply two large matrices (5000x5000) and compare speeds.

In [ ]:
# Configuration
SIZE = 5000

# --- CPU Benchmark ---
print(f"Initializing CPU arrays ({SIZE}x{SIZE})...")
a_cpu = np.random.rand(SIZE, SIZE)
b_cpu = np.random.rand(SIZE, SIZE)

print("Starting CPU Matrix Multiplication...")
start = time.time()
c_cpu = np.dot(a_cpu, b_cpu)
end = time.time()
cpu_time = end - start
print(f"CPU Time: {cpu_time:.4f} seconds")

# --- GPU Benchmark ---
print(f"\nInitializing GPU arrays ({SIZE}x{SIZE})...")
a_gpu = cp.random.rand(SIZE, SIZE)
b_gpu = cp.random.rand(SIZE, SIZE)

# Warmup pass (GPUs have initialization overhead)
cp.dot(a_gpu, b_gpu)
cp.cuda.Stream.null.synchronize() # Wait for GPU to finish

print("Starting GPU Matrix Multiplication...")
start = time.time()
c_gpu = cp.dot(a_gpu, b_gpu)
cp.cuda.Stream.null.synchronize() # CRITICAL: Wait for GPU
end = time.time()
gpu_time = end - start
print(f"GPU Time: {gpu_time:.4f} seconds")

# Speedup Calculation
print(f"\nSpeedup: {cpu_time / gpu_time:.1f}x faster")

## Understanding `cp.cuda.Stream.null.synchronize()`

In GPU programming, many operations are executed **asynchronously**. This means that when a GPU operation is launched, control may return to the CPU **before the GPU has finished executing the task**. While this improves performance, it can lead to incorrect timing measurements or unintended behavior if not handled properly.

### What Is a CUDA Stream?

A **CUDA stream** is a sequence of operations that are executed on the GPU **in order**.  
By default, CuPy uses the **default CUDA stream**, which is referred to as the *null stream*.

In CuPy:
- `cp.cuda.Stream.null` represents the **default CUDA stream**
- Operations submitted to this stream are executed sequentially on the GPU

### What Does `synchronize()` Do?
The method:
```python```
cp.cuda.Stream.null.synchronize()

<center><img alt="" src="images/APDS/cuda-stream.jpg" style="height: 400px;"/></center>

### 📝 Guided Exercise 1
**Task:** Refactor the following CPU code to run on the GPU using CuPy.

```python
import numpy as np
def calculate_hypotenuse(a, b):
    return np.sqrt(a**2 + b**2)
```

1. Create two arrays of size 10,000,000 on the GPU.
2. Calculate the hypotenuse.
3. Measure the time.

In [ ]:
# TODO: Student Implementation Here

# Solution:
# a_gpu = cp.random.random(10000000)
# b_gpu = cp.random.random(10000000)
# start = time.time()
# res = cp.sqrt(a_gpu**2 + b_gpu**2)
# cp.cuda.Stream.null.synchronize()
# print(f"Done in {time.time()-start}s")

# The Bottleneck - Data Transfer

Moving data between CPU (RAM) and GPU (VRAM) is expensive.

### The PCIe Bus
Think of the CPU and GPU as two islands connected by a bridge (PCIe bus). Computing on the island is fast, but crossing the bridge is slow.

<center><img alt="" src="images/APDS/GPU-Bottleneck.jpg" style="height: 400px;"/></center>

**Common Pitfall:** Moving data back and forth inside a loop.

In [ ]:
# Demonstrating the Overhead
size_small = 100

# CPU
a_cpu = np.random.rand(size_small, size_small)

start = time.time()
np.dot(a_cpu, a_cpu)
print(f"CPU (Small Data): {time.time() - start:.6f}s")

# GPU (Includes Transfer Time)
start = time.time()
# 1. Transfer to GPU
a_gpu = cp.asarray(a_cpu) 
# 2. Compute
res_gpu = cp.dot(a_gpu, a_gpu)
# 3. Transfer back to CPU
res_cpu = cp.asnumpy(res_gpu)
print(f"GPU (Small Data + Transfer): {time.time() - start:.6f}s")

# Insight: For small data, the overhead > computation time.

# Numba - Custom GPU Kernels

### Why Numba?
CuPy is great for matrices, but what if you have complex logic (custom formulas, loops) that isn't just a matrix multiplication?
We will use the `@vectorize` decorator to create "Universal Functions" (ufuncs) that run on the GPU.

- **CuPy**: best for quickly accelerating existing NumPy code
- **Numba**: useful for custom GPU logic

<center><img alt="" src="images/APDS/numba.png" style="height: 250px;"/></center>

### https://numba.pydata.org/

In [ ]:
from numba import vectorize, cuda
import math

# Define a function to compile for the GPU
# The signature 'float32(float32, float32)' means: Return float32, take two float32 inputs

@vectorize(['float32(float32, float32)'], target='cuda')
def gpu_function(x, y):
    # This looks like Python, but it compiles to a CUDA Kernel
    return math.sin(x) * math.cos(y) + math.exp(x / 100.0)

# Create data
n = 10000000
x = np.ones(n, dtype=np.float32)
y = np.ones(n, dtype=np.float32)

print("Running Numba CUDA Kernel...")
start = time.time()
# We can pass NumPy arrays directly; Numba handles the transfer (though manually managing is faster)
result = gpu_function(x, y)
print(f"Done in {time.time() - start:.4f}s")

### ⚠️ Important Concepts for Numba
1.  **Scalar Operations Only:** Inside the `@vectorize` function, you write logic for *one single element*. The GPU applies this to millions of elements at once.
2.  **No Python Objects:** You cannot use lists, dictionaries, or strings inside these functions. Only numbers.

# Capstone Exercise - Monte Carlo Pi

**Concept:** We can estimate the value of Pi by throwing random darts at a square board and counting how many land inside the inscribed circle.

**Task:**
1. Generate random `x` and `y` coordinates.
2. Calculate distance from center: $d = \sqrt{x^2 + y^2}$
3. Check if $d <= 1.0$.
4. Ratio of (inside / total) * 4 $\approx \pi$.

**Instructions:**
Write a solution using **CuPy** to perform this simulation with 100,000,000 points. Compare it to a pure Python implementation.

In [ ]:
# --- Student Workspace ---
# Hint: Use cp.random.rand()





In [ ]:
# --- Solution ---

def estimate_pi_gpu(num_points):
    print(f"Simulating {num_points} points on GPU...")
    start = time.time()
    
    # 1. Generate random numbers directly on GPU
    x = cp.random.rand(num_points, dtype=cp.float32)
    y = cp.random.rand(num_points, dtype=cp.float32)
    
    # 2. Compute distance (vectorized)
    dist = x**2 + y**2
    
    # 3. Count points inside circle
    inside_circle = (dist <= 1.0).sum()
    
    pi_est = 4.0 * inside_circle / num_points
    cp.cuda.Stream.null.synchronize()
    
    end = time.time()
    print(f"Estimated Pi: {pi_est}")
    print(f"Time taken: {end - start:.4f}s")
    return pi_est

estimate_pi_gpu(100000000)

# Wrap-up & Best Practices (so far)

## 🎓 Summary

1.  **Use CuPy** as your first line of defense. It's the easiest way to speed up NumPy code.
2.  **Use Numba** when you need custom element-wise logic that CuPy doesn't support.
3.  **Mind the Transfer:** Avoid `cp.asnumpy()` inside loops. Keep data on the GPU as long as possible.
4.  **Profile First:** Don't optimize code that isn't slow. Use `%time` or `time.time()`.



# Enterprise Scale: The RAPIDS Ecosystem

RAPIDS brings GPU acceleration to standard Data Science DataFrames (Pandas) and Machine Learning (Scikit-Learn).

Note for Colab Users: Recent versions of Colab have RAPIDS pre-installed. We can use the special %load_ext cudf.pandas magic command. This automatically accelerates standard Pandas code on the GPU. If an operation isn't supported on GPU, it seamlessly falls back to CPU.

<center><img alt="" src="images/APDS/Rapids-cuDf.png" style="height: 250px;"/></center>

### https://github.com/rapidsai/cudf

# Zero-Code-Change Acceleration

> This example demonstrates how to speed up Pandas without learning a new API.

In [ ]:
# Enable RAPIDS accelerator for Pandas
%load_ext cudf.pandas

In [ ]:
import pandas as pd
import numpy as np
import time

def demo_pandas_acceleration():
    # 1. Create a large synthetic dataset (10 Million rows)
    num_rows = 10_000_000
    print(f"Creating DataFrame with {num_rows} rows...")
    
    df = pd.DataFrame({
        'id': np.random.randint(0, 1000, num_rows),
        'val1': np.random.rand(num_rows),
        'val2': np.random.rand(num_rows)
    })
    
    # 2. Perform a heavy GroupBy operation
    print("Running GroupBy (Accelerated)...")
    start = time.time()
    
    # This looks like standard Pandas, but runs on GPU because of the extension loaded above
    res = df.groupby('id').agg({'val1': 'mean', 'val2': 'std'})
    
    print(f"Execution Time: {time.time() - start:.4f} seconds")
    print(f"Result shape: {res.shape}")

demo_pandas_acceleration()